In [ ]:
#04_modelling_evalutation.ipynb

In [1]:
!git clone https://github.com/Aymanberri/RealEstate_Analytics.git


Cloning into 'RealEstate_Analytics'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 192 (delta 108), reused 93 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 339.73 KiB | 9.71 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [2]:
%cd RealEstate_Analytics
!ls

/content/RealEstate_Analytics
app  architecture.md  data  notebooks  README.md  requirements.txt


---

# Notebook 04 — Modeling & Evaluation

This notebook trains and evaluates machine learning models
to predict annual rent prices for apartments in JVC, Dubai.

## Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load dataset

The csv we prepared in step 03.

In [10]:
DATA_PATH = "data/jvc_apartments_ml.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (951, 51)


,title,price,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,...,building_grouped_Westview Garden,district_JVC District 11,district_JVC District 12,district_JVC District 13,district_JVC District 14,district_JVC District 15,district_JVC District 16,district_JVC District 17,district_JVC District 18,district_Unknown
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,...,False,False,False,True,False,False,False,False,False,False
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,...,False,False,False,False,False,False,False,False,True,False
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,...,False,False,False,True,False,False,False,False,False,False
3,Converted into 2BR | Private Garden | Furnished,"140,000",yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,...,False,False,False,False,False,False,False,False,False,False
4,Spacious 1Br | Prime Location | JVC,"75,000",yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,...,False,False,False,True,False,False,False,False,False,False


In [11]:
print(df.columns)

Index(['title', 'price', 'frequency', 'bedrooms', 'bathrooms', 'area',
       'location', 'url', 'price_clean', 'price_yearly_aed', 'bedrooms_clean',
       'bathrooms_clean', 'area_clean', 'area_per_bedroom',
       'bathrooms_per_bedroom', 'log_area', 'log_price', 'building',
       'community', 'property_type_penthouse', 'property_type_townhouse',
       'property_type_villa', 'building_grouped_Binghatti Corner',
       'building_grouped_Binghatti Crest',
       'building_grouped_Binghatti Heights',
       'building_grouped_Binghatti House',
       'building_grouped_Binghatti Phantom',
       'building_grouped_Binghatti Phoenix',
       'building_grouped_Binghatti Royale',
       'building_grouped_Bloom Heights 1, Bloom Heights',
       'building_grouped_DAMAC Ghalia', 'building_grouped_Fortunato',
       'building_grouped_Imperial Tower', 'building_grouped_Laya Residences',
       'building_grouped_Other', 'building_grouped_Pearl House 2',
       'building_grouped_Reef Residence', 

## Define Train (X) and Target (y) features

In [13]:
# We must drop columns that are useless (memory usage) and derived from the `price_yearly_aed` (data leakage)
# this step is not cleaning or preprocessesing, this is why I put it here. In ML we now choose only what we need.
DROP_COLS = [
    "title",
    "price",      # Derived features cause data leakage
    "price_clean", # Derived
    "frequency",
    "url",
    "location",
    "area",
    "bedrooms",
    "bathrooms",

    "price_yearly_aed",
    "log_price" # Derived
]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y = df["price_yearly_aed"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (951, 40)
y shape: (951,)
